<a href="https://colab.research.google.com/github/lavg270994-hue/via-fluida-mas/blob/main/via_fluida_crew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vía Fluida — Fase 3: Arquitectura Multiagente (MAS) con CrewAI
**Universidad Anáhuac** · **Curso:** Inteligencia Artificial · **Profesor:** Paulo Daniel Vázquez Mora
**Proyecto:** Agentes · **Equipo 3:** Bello León Cristian - Carrizosa Dávila Carlos Andrés -
Córdova Flores Alejandro Oliver - Gutiérrez Avalos Nallely Gabriela - Vera Guevara Luis Alberto

Prototipo de un *Crew* de 4 agentes especializados que, ante un evento de congestión
(partido + lluvia + choque), (1) diagnostican la red, (2) ajustan los semáforos con
ola verde, (3) recomiendan rutas alternas sin "efecto manada" y (4) validan contra la
meta PEAS y consolidan un reporte para el centro de control.

**Cómo ejecutarlo en Colab:** *Entorno de ejecución → Ejecutar todas*.
* `VIAFLUIDA_LLM = "simulado"` (por defecto): no necesita API key. Un LLM *simulado y determinista* actúa como "cerebro" de los agentes, pero **CrewAI, los agentes, las tareas, el flujo de contexto y las 7 herramientas se ejecutan de verdad**. Todas las cifras provienen de las herramientas.
* `"openai"` / `"gemini"`: usa un LLM real; define antes la llave (`OPENAI_API_KEY` o `GEMINI_API_KEY`) en la celda de configuración.
* `"ollama"`: Llama 3.2 local (requiere `ollama serve`; normalmente se usa fuera de Colab).

**Nota:** los datos de sensores y de la red vial son sintéticos (ilustrativos).

## 0. Instalación y configuración

In [1]:
# Instalamos CrewAI
!pip -q install crewai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [2]:
# Configuración del modelo: "simulado" (sin API key), "openai", "gemini" u "ollama"
import os
os.environ["VIAFLUIDA_LLM"] = "simulado"
# os.environ["OPENAI_API_KEY"] = "..."   # solo si usas "openai"
# os.environ["GEMINI_API_KEY"] = "..."   # solo si usas "gemini"

import json
import os
import re
import warnings

os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
warnings.filterwarnings("ignore")

from crewai import Agent, BaseLLM, Crew, LLM, Process, Task
from crewai.tools import tool

## 1. Entorno simulado de la red vial (datos de sensores)
Sábado 19:30, lluvia moderada, salida de partido y choque en el tramo INT-A|INT-B. **Datos sintéticos.**

In [3]:
VEL_LIBRE_KMH = 40.0       # velocidad de flujo libre urbana de referencia
FLUJO_SATURACION = 1800    # veh/h por carril (valor típico de Webster)
PERDIDA_POR_FASE_S = 4     # tiempo perdido por fase (ámbar + todo-rojo)

# Por intersección: fases NS (norte-sur) y EO (este-oeste); por acceso:
# cola (veh), vel (km/h), espera medida (s), flujo (veh/h) y carriles.
SENSORES = {
    "INT-A": {"nombre": "Acceso norte zona estadio", "ciclo_s": 90,
              "verde_actual": {"NS": 41, "EO": 41},
              "accesos": {"N": dict(cola=38, vel=6, espera=142, flujo=2600, carriles=2),
                          "S": dict(cola=12, vel=18, espera=55, flujo=700, carriles=2),
                          "E": dict(cola=20, vel=11, espera=88, flujo=900, carriles=2),
                          "O": dict(cola=9, vel=22, espera=40, flujo=500, carriles=1)}},
    "INT-B": {"nombre": "Eje principal (aguas abajo del choque)", "ciclo_s": 90,
              "verde_actual": {"NS": 41, "EO": 41},
              "accesos": {"N": dict(cola=6, vel=24, espera=35, flujo=900, carriles=2),
                          "S": dict(cola=8, vel=21, espera=40, flujo=1000, carriles=2),
                          "E": dict(cola=11, vel=15, espera=62, flujo=1500, carriles=2),
                          "O": dict(cola=14, vel=13, espera=70, flujo=1700, carriles=2)}},
    "INT-C": {"nombre": "Alterna oriente", "ciclo_s": 90,
              "verde_actual": {"NS": 41, "EO": 41},
              "accesos": {"N": dict(cola=3, vel=30, espera=22, flujo=600, carriles=1),
                          "S": dict(cola=4, vel=29, espera=24, flujo=650, carriles=1),
                          "E": dict(cola=2, vel=33, espera=18, flujo=500, carriles=1),
                          "O": dict(cola=3, vel=31, espera=20, flujo=550, carriles=1)}},
    "INT-D": {"nombre": "Alterna sur / conector", "ciclo_s": 90,
              "verde_actual": {"NS": 41, "EO": 41},
              "accesos": {"N": dict(cola=4, vel=28, espera=26, flujo=700, carriles=1),
                          "S": dict(cola=5, vel=27, espera=28, flujo=720, carriles=1),
                          "E": dict(cola=3, vel=31, espera=21, flujo=500, carriles=1),
                          "O": dict(cola=4, vel=29, espera=25, flujo=560, carriles=1)}},
}

# Red vial: tramos entre nodos (km, vel. libre km/h, capacidad veh/h, carga base).
TRAMOS = {
    ("INT-A", "INT-B"): dict(km=1.5, vlibre=50, cap=3600, base=1800),
    ("INT-B", "PERIF"): dict(km=1.2, vlibre=50, cap=3600, base=1800),
    ("INT-A", "INT-C"): dict(km=1.1, vlibre=40, cap=1500, base=500),
    ("INT-C", "INT-D"): dict(km=1.6, vlibre=40, cap=1500, base=450),
    ("INT-D", "PERIF"): dict(km=0.9, vlibre=40, cap=1500, base=400),
    ("INT-B", "INT-D"): dict(km=1.0, vlibre=35, cap=1200, base=300),
}


def _tramo(a, b):
    """Devuelve el tramo (no dirigido) entre dos nodos, o None."""
    return TRAMOS.get((a, b)) or TRAMOS.get((b, a))

## 2. Herramientas personalizadas (`tools`)
Siete herramientas con el decorador `@tool`. Hacen todos los cálculos (TTI, Webster, ola verde, BPR y validación PEAS), así las cifras son reproducibles y no las inventa el LLM.

In [4]:
@tool("leer_sensores_interseccion")
def leer_sensores_interseccion(id_interseccion: str) -> str:
    """Lee los sensores de una intersección (INT-A..INT-D) o de todas ('TODAS').
    Devuelve JSON con cola, velocidad, espera y flujo por acceso."""
    if id_interseccion.upper() == "TODAS":
        return json.dumps(SENSORES, ensure_ascii=False)
    datos = SENSORES.get(id_interseccion.upper())
    return json.dumps(datos, ensure_ascii=False) if datos else "ERROR: id inválido"


@tool("calcular_indice_congestion")
def calcular_indice_congestion(id_interseccion: str) -> str:
    """Calcula el Travel Time Index (TTI = v_libre / v_actual) por acceso y el
    TTI ponderado por flujo de la intersección; clasifica la severidad."""
    inter = SENSORES.get(id_interseccion.upper())
    if not inter:
        return "ERROR: id inválido"
    tti, flujo_total, suma = {}, 0, 0.0
    for acc, d in inter["accesos"].items():
        tti[acc] = round(VEL_LIBRE_KMH / max(d["vel"], 1), 2)
        suma += tti[acc] * d["flujo"]
        flujo_total += d["flujo"]
    ponderado = round(suma / flujo_total, 2)
    nivel = ("fluido" if ponderado < 1.3 else "moderado" if ponderado < 2.0
             else "severo" if ponderado < 3.0 else "critico")
    peor = max(inter["accesos"], key=lambda a: inter["accesos"][a]["cola"])
    return json.dumps({"interseccion": id_interseccion.upper(), "tti_por_acceso": tti,
                       "tti_ponderado": ponderado, "severidad": nivel,
                       "acceso_critico": peor,
                       "cola_max_veh": inter["accesos"][peor]["cola"]})


def _demora_uniforme(c, g, flujo, cap_sat, periodo_s=900):
    """Demora media por vehículo (s): término uniforme de Webster + término
    determinista de sobresaturación. Modelo simplificado, no medición."""
    lam = g / c
    x = flujo / (cap_sat * lam)  # grado de saturación
    d1 = c * (1 - lam) ** 2 / (2 * (1 - lam * min(x, 0.98)))
    d2 = (x - 1) * periodo_s / 2 if x > 1 else 0.0
    return d1 + d2


def _plan_semaforico(id_interseccion, ciclo_comun_s=0):
    """Lógica de Webster (compartida por dos herramientas). Devuelve dict."""
    inter = SENSORES.get(id_interseccion.upper())
    if not inter:
        return None
    acc = inter["accesos"]
    # Razón de flujo crítica por fase: y = q / s (se toma el acceso crítico)
    y = {f: max(acc[a]["flujo"] / (FLUJO_SATURACION * acc[a]["carriles"])
                for a in accs) for f, accs in {"NS": "NS", "EO": "EO"}.items()}
    Y, L = y["NS"] + y["EO"], 2 * PERDIDA_POR_FASE_S
    sobresaturada = Y >= 0.95
    c0 = 120 if sobresaturada else (1.5 * L + 5) / (1 - Y)
    c = int(round(min(max(c0, 60), 120)))  # acotado a [60, 120] s
    if ciclo_comun_s > 0:
        c = int(ciclo_comun_s)  # ciclo impuesto por coordinación
    g_ef = c - L
    g = {f: max(10, round(g_ef * y[f] / Y)) for f in y}  # verde mínimo 10 s

    def demora(ciclo, verdes):
        tot_q = tot_d = 0
        for a, d in acc.items():
            f = "NS" if a in "NS" else "EO"
            dem = _demora_uniforme(ciclo, verdes[f], d["flujo"],
                                   FLUJO_SATURACION * d["carriles"])
            tot_d += dem * d["flujo"]
            tot_q += d["flujo"]
        return tot_d / tot_q

    return {"interseccion": id_interseccion.upper(),
            "Y": round(Y, 2), "sobresaturada": sobresaturada,
            "ciclo_actual_s": inter["ciclo_s"],
            "verde_actual": inter["verde_actual"],
            "ciclo_propuesto_s": c, "verde_propuesto": g,
            "demora_modelo_actual_s": round(demora(inter["ciclo_s"], inter["verde_actual"]), 1),
            "demora_modelo_propuesta_s": round(demora(c, g), 1),
            "flujo_total_vph": sum(d["flujo"] for d in acc.values())}


@tool("calcular_plan_semaforico")
def calcular_plan_semaforico(id_interseccion: str, ciclo_comun_s: int = 0) -> str:
    """Calcula el ciclo óptimo de Webster y el reparto de verdes (NS/EO) para
    una intersección y estima la demora antes/después con un modelo simple.
    Si 'ciclo_comun_s' > 0 se fuerza ese ciclo (requisito para una ola verde:
    las intersecciones coordinadas deben compartir el mismo ciclo)."""
    plan = _plan_semaforico(id_interseccion, ciclo_comun_s)
    return json.dumps(plan) if plan else "ERROR: id inválido"


@tool("resumir_demora_ponderada")
def resumir_demora_ponderada(ids_csv: str) -> str:
    """Demora media ponderada por flujo (antes/después) de varias intersecciones
    separadas por coma; la PRIMERA es la más crítica y fija el ciclo común.
    Evita que el LLM haga la aritmética a mano."""
    ids = [i.strip() for i in ids_csv.split(",")]
    primero = _plan_semaforico(ids[0])
    if not primero:
        return "ERROR: id inválido"
    planes = [primero] + [_plan_semaforico(i, primero["ciclo_propuesto_s"])
                          for i in ids[1:]]
    q = sum(p["flujo_total_vph"] for p in planes)
    antes = sum(p["demora_modelo_actual_s"] * p["flujo_total_vph"] for p in planes) / q
    despues = sum(p["demora_modelo_propuesta_s"] * p["flujo_total_vph"] for p in planes) / q
    return json.dumps({"ciclo_comun_s": primero["ciclo_propuesto_s"],
                       "demora_media_antes_s": round(antes, 1),
                       "demora_media_despues_s": round(despues, 1)})


@tool("calcular_offset_ola_verde")
def calcular_offset_ola_verde(origen: str, destino: str, ciclo_s: int = 90,
                              vel_progresion_kmh: float = 40.0) -> str:
    """Calcula el desfase (offset, s) entre dos intersecciones vecinas para
    coordinar una 'ola verde': offset = (distancia / velocidad) mod ciclo."""
    t = _tramo(origen.upper(), destino.upper())
    if not t:
        return "ERROR: tramos no adyacentes"
    viaje_s = t["km"] * 1000 / (vel_progresion_kmh / 3.6)
    return json.dumps({"origen": origen, "destino": destino, "km": t["km"],
                       "tiempo_viaje_s": round(viaje_s, 1),
                       "offset_s": round(viaje_s % ciclo_s, 1)})


def _rutas(origen, destino, ruta=None):
    """Enumera rutas simples (DFS) entre dos nodos de la red."""
    ruta = (ruta or []) + [origen]
    if origen == destino:
        return [ruta]
    vecinos = sorted({b if a == origen else a for (a, b) in TRAMOS if origen in (a, b)})
    return [r for v in vecinos if v not in ruta for r in _rutas(v, destino, ruta)]


def _t_bpr(km, vlibre, cap, carga):
    """Tiempo de viaje (s) con la función de desempeño tipo BPR."""
    t0 = km / vlibre * 3600
    return t0 * (1 + 0.15 * (carga / cap) ** 4)


@tool("simular_asignacion_rutas")
def simular_asignacion_rutas(origen: str, destino: str, demanda_vph: int,
                             tramo_incidente: str, reduccion_capacidad: float) -> str:
    """Compara 3 escenarios para 'demanda_vph' conductores: (A) todos por la ruta
    habitual, (B) todos a la alterna más rápida (efecto manada) y (C) asignación
    coordinada por incrementos. tramo_incidente='INT-A|INT-B'; la capacidad del
    tramo se reduce en 'reduccion_capacidad' (0.5 = 50 %)."""
    a, b = tramo_incidente.split("|")
    cap_inc = {frozenset((a, b)): 1 - reduccion_capacidad}
    rutas = _rutas(origen, destino)
    tramos_de = lambda r: [frozenset(p) for p in zip(r, r[1:])]

    def t_tramo(par, cargas):
        n1, n2 = tuple(par)
        t = _tramo(n1, n2)
        return _t_bpr(t["km"], t["vlibre"], t["cap"] * cap_inc.get(par, 1.0),
                      t["base"] + cargas.get(par, 0))

    def t_ruta(r, cargas):
        return sum(t_tramo(p, cargas) for p in tramos_de(r))

    def evalua(reparto):  # reparto: {indice: veh/h}
        cargas = {}
        for i, q in reparto.items():
            for p in tramos_de(rutas[i]):
                cargas[p] = cargas.get(p, 0) + q
        tiempos = {i: t_ruta(rutas[i], cargas) for i in reparto if reparto[i] > 0}
        medio = sum(reparto[i] * tiempos[i] for i in tiempos) / demanda_vph
        return medio, tiempos

    sin_carga = {}
    habitual = min(range(len(rutas)), key=lambda i: sum(
        _tramo(*tuple(p))["km"] / _tramo(*tuple(p))["vlibre"] for p in tramos_de(rutas[i])))
    inc = frozenset((a, b))  # la 'mejor alterna' debe evitar el tramo del incidente
    sin_inc = [i for i in range(len(rutas)) if inc not in tramos_de(rutas[i])]
    mejor_alt = min(sin_inc, key=lambda i: t_ruta(rutas[i], sin_carga))

    # Escenario C: asignación incremental (20 paquetes) a la ruta más rápida
    reparto, paso = {i: 0 for i in range(len(rutas))}, demanda_vph / 20
    for _ in range(20):
        cargas = {}
        for i, q in reparto.items():
            for p in tramos_de(rutas[i]):
                cargas[p] = cargas.get(p, 0) + q
        k = min(range(len(rutas)), key=lambda i: t_ruta(rutas[i], cargas))
        reparto[k] += paso

    esc = {}
    for nombre, rep in {"A_sin_intervencion": {habitual: demanda_vph},
                        "B_efecto_manada": {mejor_alt: demanda_vph},
                        "C_asignacion_coordinada": reparto}.items():
        medio, tiempos = evalua({i: rep.get(i, 0) for i in range(len(rutas))})
        esc[nombre] = {"tiempo_medio_min": round(medio / 60, 1),
                       "reparto_pct": {" > ".join(rutas[i]): round(100 * q / demanda_vph)
                                       for i, q in rep.items() if q > 0},
                       "tiempo_por_ruta_min": {" > ".join(rutas[i]): round(t / 60, 1)
                                               for i, t in tiempos.items()}}
    return json.dumps(esc, ensure_ascii=False)


@tool("validar_contra_peas")
def validar_contra_peas(demora_actual_s: float, demora_propuesta_s: float,
                        viaje_actual_min: float, viaje_propuesto_min: float) -> str:
    """Compara contra la métrica principal PEAS (reducción >= 15 % del tiempo
    de espera y de viaje) y devuelve el porcentaje y si se cumple la meta."""
    r_espera = 100 * (demora_actual_s - demora_propuesta_s) / demora_actual_s
    r_viaje = 100 * (viaje_actual_min - viaje_propuesto_min) / viaje_actual_min
    return json.dumps({"reduccion_espera_pct": round(r_espera, 1),
                       "reduccion_viaje_pct": round(r_viaje, 1),
                       "cumple_meta_15pct": bool(r_espera >= 15 and r_viaje >= 15)})

## 3. Modelo de lenguaje del Crew
Selector `VIAFLUIDA_LLM`: `simulado` (subclase de `BaseLLM`, formato ReAct de CrewAI, sin red), `openai`, `gemini` u `ollama`.

In [5]:
def _json_en_texto(texto):
    """Extrae todos los objetos JSON contenidos en un texto (salidas de tareas previas)."""
    dec, objetos, pos = json.JSONDecoder(), [], 0
    while (ini := texto.find("{", pos)) != -1:
        try:
            obj, n = dec.raw_decode(texto[ini:])
            objetos.append(obj)
            pos = ini + n
        except ValueError:
            pos = ini + 1
    return objetos


def _obs_json(texto):
    """Primer objeto JSON de una observación (CrewAI puede anexar recordatorios)."""
    objetos = _json_en_texto(texto)
    return objetos[0] if objetos else {}


class LLMSimulado(BaseLLM):
    """LLM determinista para demostrar el Crew sin API key ni red.

    Responde en el formato ReAct que CrewAI espera (Thought / Action /
    Action Input / Final Answer). CrewAI interpreta esas respuestas, ejecuta
    las herramientas REALES y le devuelve las observaciones; este 'cerebro'
    solo decide qué herramienta llamar y redacta la respuesta final a partir
    de las observaciones y del context recibido. Todas las cifras provienen
    de las herramientas, nunca de este guion.
    """

    llm_type: str = "simulado"

    def supports_function_calling(self) -> bool:
        return False  # obliga al formato de texto ReAct de CrewAI

    def supports_stop_words(self) -> bool:
        return False

    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        rol = getattr(from_agent, "role", "")

        # Observaciones que CrewAI ya devolvió de las herramientas en esta tarea
        obs = []
        for m in messages:
            c = m.get("content") or ""
            if not isinstance(c, str):
                continue
            if m.get("role") == "assistant" and "\nObservation: " in c:
                obs.append(c.split("\nObservation: ", 1)[1].strip())
            elif m.get("role") == "user" and c.startswith("Observation: "):
                obs.append(c[len("Observation: "):].strip())

        # Context recibido de tareas previas
        texto_usuario = "\n".join(m.get("content") or "" for m in messages
                                  if m.get("role") == "user" and isinstance(m.get("content"), str))
        ctx = []
        if "context you're working with:" in texto_usuario:
            ctx = _json_en_texto(texto_usuario.split("context you're working with:", 1)[1])

        def accion(pensamiento, herramienta, args):
            return (f"Thought: {pensamiento}\nAction: {herramienta}\n"
                    f"Action Input: {json.dumps(args, ensure_ascii=False)}")

        def final(pensamiento, respuesta):
            return f"Thought: {pensamiento}\nFinal Answer: {respuesta}"

        if rol.startswith("Analista"):
            return self._analista(obs, accion, final)
        if rol.startswith("Optimizador"):
            return self._optimizador(obs, ctx, accion, final)
        if rol.startswith("Asesor"):
            return self._asesor(obs, ctx, texto_usuario, accion, final)
        if rol.startswith("Supervisor"):
            return self._supervisor(obs, ctx, accion, final)
        return final("Rol no contemplado por el guion.", "{}")

    # --- Agente 1: Analista de Percepción Vial ---------------------------------
    def _analista(self, obs, accion, final):
        ids = list(SENSORES)
        if not obs:
            return accion("Primero leo los sensores de toda la red y después calculo "
                          "el TTI de cada intersección.",
                          "leer_sensores_interseccion", {"id_interseccion": "TODAS"})
        if len(obs) <= len(ids):
            sig = ids[len(obs) - 1]
            return accion(f"Calculo el índice de congestión (TTI) de {sig}.",
                          "calcular_indice_congestion", {"id_interseccion": sig})
        indices = [_obs_json(o) for o in obs[1:]]
        indices.sort(key=lambda d: d["tti_ponderado"], reverse=True)
        lista = [{"id": d["interseccion"], "tti_ponderado": d["tti_ponderado"],
                  "severidad": d["severidad"], "acceso_critico": d["acceso_critico"],
                  "cola_max_veh": d["cola_max_veh"]} for d in indices]
        graves = [d for d in lista if d["severidad"] in ("severo", "critico")]
        resto = [d["id"] for d in lista if d not in graves]
        resumen = ("; ".join(f"{d['id']} en estado {d['severidad']} (TTI {d['tti_ponderado']})"
                             for d in graves)
                   + f". {', '.join(resto)} con margen para absorber desvíos.")
        return final("Ordené las intersecciones por TTI ponderado de mayor a menor; "
                     "no propongo acciones.",
                     json.dumps({"intersecciones": lista, "resumen": resumen},
                                ensure_ascii=False))

    # --- Agente 2: Optimizador Semafórico Adaptativo ---------------------------
    def _optimizador(self, obs, ctx, accion, final):
        diag = next((c for c in ctx if isinstance(c.get("intersecciones"), list)), {})
        objetivo = [d["id"] for d in diag.get("intersecciones", [])
                    if d.get("severidad") in ("severo", "critico")]
        n = len(objetivo)
        if n == 0:
            return final("El diagnóstico no reporta intersecciones severas o críticas.",
                         json.dumps({"planes": [], "olas_verdes": []}))
        if len(obs) < n:  # un plan por intersección; la primera fija el ciclo común
            i = len(obs)
            if i == 0:
                return accion(f"{objetivo[0]} es la más crítica: calculo su plan primero "
                              "para fijar el ciclo común.",
                              "calcular_plan_semaforico",
                              {"id_interseccion": objetivo[0], "ciclo_comun_s": 0})
            ciclo = _obs_json(obs[0])["ciclo_propuesto_s"]
            return accion(f"Para la ola verde, {objetivo[i]} debe usar el mismo ciclo "
                          f"de {ciclo} s.",
                          "calcular_plan_semaforico",
                          {"id_interseccion": objetivo[i], "ciclo_comun_s": ciclo})
        planes = [_obs_json(o) for o in obs[:n]]
        ciclo = planes[0]["ciclo_propuesto_s"]
        pares = [(objetivo[k], objetivo[k + 1]) for k in range(n - 1)
                 if _tramo(objetivo[k], objetivo[k + 1])]
        if len(obs) < n + len(pares):
            o, d = pares[len(obs) - n]
            return accion(f"Calculo el desfase de la ola verde {o} -> {d}.",
                          "calcular_offset_ola_verde",
                          {"origen": o, "destino": d, "ciclo_s": ciclo,
                           "vel_progresion_kmh": 40.0})
        if len(obs) == n + len(pares):
            return accion("Uso la herramienta para la demora media ponderada; "
                          "no la calculo a mano.",
                          "resumir_demora_ponderada", {"ids_csv": ",".join(objetivo)})
        olas = [_obs_json(o) for o in obs[n:n + len(pares)]]
        resumen = _obs_json(obs[-1])
        salida = {"planes": [{"id": p["interseccion"],
                              "ciclo_propuesto_s": p["ciclo_propuesto_s"],
                              "verde_propuesto": p["verde_propuesto"],
                              "demora_modelo_actual_s": p["demora_modelo_actual_s"],
                              "demora_modelo_propuesta_s": p["demora_modelo_propuesta_s"]}
                             for p in planes],
                  "olas_verdes": [{"origen": w["origen"], "destino": w["destino"],
                                   "offset_s": w["offset_s"]} for w in olas],
                  "demora_media_ponderada_antes_s": resumen["demora_media_antes_s"],
                  "demora_media_ponderada_despues_s": resumen["demora_media_despues_s"]}
        return final("Planes con ciclo común calculados; las cifras salen de las herramientas.",
                     json.dumps(salida, ensure_ascii=False))

    # --- Agente 3: Asesor de Rutas Alternas y Anti-Manada ----------------------
    def _asesor(self, obs, ctx, texto_usuario, accion, final):
        if not obs:
            m = re.search(r"asignación de (\d+) veh/h que salen de (\S+) hacia (\S+) "
                          r"con incidente en el tramo (\S+)", texto_usuario)
            demanda, origen, destino, tramo = m.groups()
            return accion("Necesito comparar los tres escenarios; no basta con recomendar "
                          "'la mejor ruta' a todos.",
                          "simular_asignacion_rutas",
                          {"origen": origen, "destino": destino.rstrip("."),
                           "demanda_vph": int(demanda), "tramo_incidente": tramo,
                           "reduccion_capacidad": 0.5})
        esc = _obs_json(obs[-1])
        reparto = {r: int(5 * round(p / 5))
                   for r, p in esc["C_asignacion_coordinada"]["reparto_pct"].items()}
        diag = next((c for c in ctx if isinstance(c.get("intersecciones"), list)), {})
        criticas = {d["id"] for d in diag.get("intersecciones", [])
                    if d.get("severidad") == "critico"}
        severas = {d["id"] for d in diag.get("intersecciones", [])
                   if d.get("severidad") == "severo"}
        a = esc["A_sin_intervencion"]["tiempo_medio_min"]
        b = esc["B_efecto_manada"]["tiempo_medio_min"]
        c = esc["C_asignacion_coordinada"]["tiempo_medio_min"]
        cruces = sorted({n for r in reparto for n in r.split(" > ")[1:] if n in criticas | severas})
        riesgo = (f"Si todos toman la alterna, el tiempo medio sube a {b} min "
                  f"(sin intervenir: {a} min); repartir lo baja a {c} min. "
                  + (f"Rutas recomendadas cruzan intersecciones graves (además del origen): "
                     f"{', '.join(cruces)}." if cruces else
                     "Ninguna ruta cruza intersecciones críticas además del origen."))
        salida = {"reparto_recomendado_pct": reparto,
                  "tiempo_por_ruta_min": esc["C_asignacion_coordinada"]["tiempo_por_ruta_min"],
                  "tiempo_medio_min": {"A_sin_intervencion": a, "B_efecto_manada": b,
                                       "C_asignacion_coordinada": c},
                  "riesgo_manada": riesgo,
                  "mensaje_panel": "CHOQUE EN EJE PRINCIPAL. ALTERNA ORIENTE-SUR HABILITADA."}
        return final("El escenario B empeora el tiempo medio; recomiendo el reparto del C.",
                     json.dumps(salida, ensure_ascii=False))

    # --- Agente 4: Supervisor y Sintetizador -----------------------------------
    def _supervisor(self, obs, ctx, accion, final):
        diag = next((c for c in ctx if isinstance(c.get("intersecciones"), list)), {})
        sem = next((c for c in ctx if "planes" in c), {})
        rut = next((c for c in ctx if "reparto_recomendado_pct" in c), {})
        tm = rut.get("tiempo_medio_min", {})
        if not obs:
            return accion("Valido la meta PEAS con las cifras de las tareas 2 y 3.",
                          "validar_contra_peas",
                          {"demora_actual_s": sem["demora_media_ponderada_antes_s"],
                           "demora_propuesta_s": sem["demora_media_ponderada_despues_s"],
                           "viaje_actual_min": tm["A_sin_intervencion"],
                           "viaje_propuesto_min": tm["C_asignacion_coordinada"]})
        val = next(v for v in map(_obs_json, obs) if "cumple_meta_15pct" in v)
        delegado = len(obs) > 1  # ya se pidió una ronda de corrección (máximo una)
        if not val["cumple_meta_15pct"] and not delegado:
            return accion("La meta PEAS no se cumple: delego el recálculo al Optimizador.",
                          "Delegate work to coworker",
                          {"task": "Recalcula el plan semafórico buscando mayor reducción de demora.",
                           "context": json.dumps(val), "coworker": "Optimizador Semafórico Adaptativo"})
        # Consistencia: rutas que cruzan intersecciones sin mejora semafórica
        mejora = {p["id"]: p["demora_modelo_propuesta_s"] < p["demora_modelo_actual_s"]
                  for p in sem.get("planes", [])}
        sev = {d["id"]: d for d in diag.get("intersecciones", [])}
        sin_mejora = sorted({n for r in rut.get("reparto_recomendado_pct", {})
                             for n in r.split(" > ")
                             if sev.get(n, {}).get("severidad") in ("severo", "critico")
                             and not mejora.get(n, False)})
        pensamiento = ("Meta cumplida (>=15 %) y sin contradicciones entre semáforos y rutas; "
                       "no es necesario delegar." if val["cumple_meta_15pct"] and not sin_mejora
                       else "Se detectaron pendientes; quedan registrados en riesgos.")
        r = [f"# Reporte Vía Fluida - {ENTRADAS['hora']}, {ENTRADAS['clima']}", "",
             "## 1. Resumen ejecutivo",
             f"{ENTRADAS['evento']} (tramo {ENTRADAS['tramo_incidente']}, capacidad -50 %). "
             f"Con las acciones propuestas, la demora estimada en las intersecciones intervenidas "
             f"baja de {sem['demora_media_ponderada_antes_s']} s a "
             f"{sem['demora_media_ponderada_despues_s']} s (-{val['reduccion_espera_pct']} %) "
             f"y el tiempo medio {ENTRADAS['origen']} -> {ENTRADAS['destino']} de "
             f"{tm['A_sin_intervencion']} a {tm['C_asignacion_coordinada']} min "
             f"(-{val['reduccion_viaje_pct']} %). Meta PEAS (>= 15 %): "
             f"{'se cumple' if val['cumple_meta_15pct'] else 'NO se cumple'} según el modelo.", "",
             "## 2. Diagnóstico", "| Intersección | TTI | Severidad | Acceso crítico (cola) |",
             "|---|---|---|---|"]
        r += [f"| {d['id']} | {d['tti_ponderado']} | {SEVERIDAD_TXT.get(d['severidad'], d['severidad'])} | "
              f"{d['acceso_critico']} ({d['cola_max_veh']} veh) |"
              for d in diag.get("intersecciones", [])]
        r += ["", "## 3. Acciones semafóricas"]
        for p in sem.get("planes", []):
            act = SENSORES[p["id"]]
            v = p["verde_propuesto"]
            r.append(f"- {p['id']}: ciclo {act['ciclo_s']} -> {p['ciclo_propuesto_s']} s; "
                     f"verde NS {act['verde_actual']['NS']} -> {v['NS']} s, "
                     f"EO {act['verde_actual']['EO']} -> {v['EO']} s.")
        for w in sem.get("olas_verdes", []):
            r.append(f"- Ola verde {w['origen']} -> {w['destino']}: offset {w['offset_s']} s.")
        r += ["", "## 4. Rutas y mensaje a conductores"]
        t_ruta = rut.get("tiempo_por_ruta_min", {})
        r += [f"- {pct} % por {ruta} ({t_ruta.get(ruta, 'n/d')} min)."
              for ruta, pct in rut.get("reparto_recomendado_pct", {}).items()]
        r += [f"- Efecto manada: si el 100 % tomara la alterna, {tm['B_efecto_manada']} min.",
              f"- Panel: \"{rut.get('mensaje_panel', '')}\"", "",
              "## 5. Validación PEAS", "| Métrica | Antes | Después | Reducción | Meta |",
              "|---|---|---|---|---|",
              f"| Demora media ponderada (s) | {sem['demora_media_ponderada_antes_s']} | "
              f"{sem['demora_media_ponderada_despues_s']} | {val['reduccion_espera_pct']} % | >= 15 % |",
              f"| Tiempo medio de viaje (min) | {tm['A_sin_intervencion']} | "
              f"{tm['C_asignacion_coordinada']} | {val['reduccion_viaje_pct']} % | >= 15 % |",
              "- Consistencia semáforos-rutas: " + (
                  "las rutas recomendadas solo cruzan intersecciones que el plan mejora "
                  f"({', '.join(sorted(k for k, v in mejora.items() if v))}). "
                  "No fue necesario delegar." if val["cumple_meta_15pct"] and not sin_mejora
                  else "hay pendientes; ver riesgos."), "",
              "## 6. Riesgos y siguientes pasos",
              "- Cifras de un modelo simplificado con datos sintéticos: validar en simulación (SUMO) y piloto."]
        moderadas = [f"{d['id']} (TTI {d['tti_ponderado']})" for d in diag.get("intersecciones", [])
                     if d["severidad"] == "moderado"]
        if moderadas:
            r.append(f"- Vigilar {', '.join(moderadas)} por posible saturación de la alterna.")
        if sin_mejora:
            r.append(f"- Rutas que cruzan intersecciones sin mejora semafórica: {', '.join(sin_mejora)}.")
        if delegado:
            r.append("- Se delegó una ronda de recálculo al Optimizador; respuesta: "
                     + obs[-1][:200].replace("\n", " "))
        r.append("- Reevaluar cada 5 min; restaurar planes al despejar el choque.")
        return final(pensamiento, "\n".join(r))


SEVERIDAD_TXT = {"critico": "Crítico", "severo": "Severo", "moderado": "Moderado",
                 "fluido": "Fluido"}


def crear_llm():
    """Devuelve el LLM según VIAFLUIDA_LLM (por defecto, 'simulado')."""
    modo = os.getenv("VIAFLUIDA_LLM", "simulado").lower()
    if modo == "simulado":
        return LLMSimulado(model="simulado-viafluida")
    if modo == "ollama":
        return LLM(model="ollama/llama3.2", base_url="http://localhost:11434",
                   temperature=0.2)
    modelos = {"openai": "gpt-4o-mini", "gemini": "gemini/gemini-2.0-flash"}
    if modo not in modelos:
        raise ValueError(f"VIAFLUIDA_LLM no válido: {modo}")
    return LLM(model=os.getenv("VIAFLUIDA_MODEL", modelos[modo]), temperature=0.2)


llm = crear_llm()

## 4. Agentes especializados (`Agent`)
Cuatro agentes con `role`, `goal`, `backstory`, `tools` y `allow_delegation` (ver Bloque B del documento). Solo el Supervisor puede delegar.

In [6]:
analista = Agent(
    role="Analista de Percepción Vial",
    goal="Diagnosticar con precisión el estado de congestión de cada intersección "
         "a partir de los sensores, sin proponer soluciones.",
    backstory="Ingeniero de tránsito de la zona metropolitana de Puebla con diez "
              "años interpretando conteos de cámaras y lazos inductivos. Es "
              "metódico y desconfía de lo que no está respaldado por una medición.",
    tools=[leer_sensores_interseccion, calcular_indice_congestion],
    allow_delegation=False, llm=llm, verbose=True, max_iter=8)

optimizador = Agent(
    role="Optimizador Semafórico Adaptativo",
    goal="Proponer ciclos, repartos de verde y desfases de ola verde que reduzcan "
         "la demora, priorizando las intersecciones más críticas.",
    backstory="Especialista en control de señales formado en estrategias "
              "adaptativas descentralizadas tipo SURTRAC. Resuelve cada cruce con "
              "Webster y luego verifica que el resultado sea coherente con sus vecinos.",
    tools=[calcular_plan_semaforico, calcular_offset_ola_verde, resumir_demora_ponderada],
    allow_delegation=False, llm=llm, verbose=True, max_iter=10)

asesor_rutas = Agent(
    role="Asesor de Rutas Alternas y Anti-Manada",
    goal="Recomendar rutas alternas repartiendo la demanda entre ellas para que "
         "ninguna se sature por el desvío masivo de conductores.",
    backstory="Analista de movilidad que estudió por qué las apps de navegación "
              "empeoran la congestión cuando todos reciben la misma recomendación. "
              "Piensa en porcentajes de reparto, no en una única mejor ruta.",
    tools=[simular_asignacion_rutas],
    allow_delegation=False, llm=llm, verbose=True, max_iter=8)

supervisor = Agent(
    role="Supervisor y Sintetizador de Movilidad",
    goal="Integrar los resultados en un reporte accionable para el centro de "
         "control, validado contra las metas PEAS y libre de contradicciones.",
    backstory="Coordinador del centro de control de movilidad urbana. Redacta "
              "decisiones claras para operadores y exige consistencia entre las "
              "recomendaciones de semáforos y las de rutas antes de aprobarlas.",
    tools=[validar_contra_peas],
    # Único agente con delegación: solo ante incumplimiento de la meta o conflicto
    allow_delegation=True, llm=llm, verbose=True, max_iter=10)

## 5. Tareas (`Task`) y flujo de información
`T1 → T2` y `T1 → T3` (el diagnóstico alimenta a semáforos y a rutas); `T1, T2, T3 → T4` (el Supervisor valida e integra mediante `context`).

In [7]:
t_diagnostico = Task(
    description=(
        "Evento: {evento}. Hora: {hora}. Clima: {clima}. Con tus herramientas, "
        "consulta los sensores y calcula el TTI de TODAS las intersecciones "
        "({intersecciones}). Clasifica la severidad, identifica el acceso crítico "
        "de cada una y ordénalas de la más a la menos crítica. No propongas acciones."),
    expected_output=(
        "JSON con la lista 'intersecciones' ordenada por criticidad; cada elemento: "
        "id, tti_ponderado, severidad, acceso_critico, cola_max_veh; y una clave "
        "'resumen' de máximo 2 frases."),
    agent=analista)

t_semaforos = Task(
    description=(
        "Usando el diagnóstico, calcula el plan semafórico propuesto para las "
        "intersecciones con severidad 'severo' o 'critico', empezando por la más "
        "crítica. Para coordinar una ola verde, las vecinas deben usar el MISMO "
        "ciclo: reutiliza el ciclo de la más crítica como 'ciclo_comun_s' y luego "
        "calcula el offset. Para la demora media ponderada por flujo usa la "
        "herramienta resumir_demora_ponderada (no calcules a mano)."),
    expected_output=(
        "JSON con 'planes' (id, ciclo_propuesto_s, verde_propuesto NS/EO, "
        "demora_modelo_actual_s, demora_modelo_propuesta_s), 'olas_verdes' "
        "(origen, destino, offset_s) y 'demora_media_ponderada_antes/despues_s'."),
    agent=optimizador, context=[t_diagnostico])

t_rutas = Task(
    description=(
        "Simula la asignación de {demanda_vph} veh/h que salen de {origen} hacia "
        "{destino} con incidente en el tramo {tramo_incidente} (capacidad -50 %). "
        "Compara los tres escenarios y recomienda el reparto porcentual por ruta "
        "(múltiplos de 5 %). Explica por qué se evita el efecto manada y señala si "
        "alguna ruta cruza una intersección calificada como crítica."),
    expected_output=(
        "JSON con 'reparto_recomendado_pct' por ruta, 'tiempo_medio_min' de los "
        "escenarios A/B/C, 'riesgo_manada' (texto) y 'mensaje_panel' (<= 80 "
        "caracteres) para los paneles de mensaje variable."),
    agent=asesor_rutas, context=[t_diagnostico])

t_reporte = Task(
    description=(
        "Consolida los resultados de las tres tareas previas. Valida con tu "
        "herramienta la meta PEAS (>=15 % de reducción de espera y de viaje). "
        "Verifica que ninguna ruta recomendada atraviese una intersección que el "
        "plan semafórico deje sin mejora. Si la meta no se cumple o hay "
        "contradicciones, DELEGA la corrección al agente responsable y reintegra."),
    expected_output=(
        "Reporte en Markdown para el centro de control con: 1) Resumen ejecutivo, "
        "2) Diagnóstico, 3) Acciones semafóricas, 4) Rutas y mensaje a conductores, "
        "5) Validación PEAS (tabla con % de reducción), 6) Riesgos y siguientes "
        "pasos. Máximo 350 palabras."),
    agent=supervisor, context=[t_diagnostico, t_semaforos, t_rutas],
    output_file="reporte_via_fluida.md")

## 6. Log de razonamiento
`step_callback` y `task_callback` de CrewAI escriben cada Thought / herramienta / salida / respuesta final en `log_via_fluida.txt`.

In [8]:
LOG_PASOS = "log_via_fluida.txt"
_contador = {"herramientas": 0, "delegaciones": 0}


def registrar_paso(paso):
    """step_callback: CrewAI lo invoca tras cada paso (AgentAction o AgentFinish).
    Escribe Thought / Action / Action Input / Observation / Final Answer en texto plano."""
    with open(LOG_PASOS, "a", encoding="utf-8") as f:
        pensamiento = (getattr(paso, "thought", "") or "").strip()
        pensamiento = re.sub(r"^(Thought:\s*)+", "", pensamiento)
        if pensamiento:
            f.write(f"Thought: {pensamiento}\n")
        if hasattr(paso, "tool"):  # AgentAction: el agente usó una herramienta
            _contador["herramientas"] += 1
            if "coworker" in paso.tool.lower():
                _contador["delegaciones"] += 1
            f.write(f"## Using tool: {paso.tool}\n## Tool Input: {paso.tool_input}\n")
            salida = str(getattr(paso, "result", ""))
            # CrewAI a veces anexa al resultado un recordatorio con la lista de
            # herramientas; en el log solo se conserva la salida de la herramienta.
            salida = salida.split("\n\n\nYou ONLY have access", 1)[0].strip()
            f.write(f"## Tool Output: {salida}\n")
        elif hasattr(paso, "output"):  # AgentFinish: respuesta final de la tarea
            f.write(f"## Final Answer:\n{paso.output}\n")


def registrar_tarea(salida):
    """task_callback: escribe la respuesta final de cada tarea y marca su cierre."""
    with open(LOG_PASOS, "a", encoding="utf-8") as f:
        f.write(f"## Final Answer:\n{salida.raw}\n")
        f.write(f"======== Fin de tarea · agente: {salida.agent} ========\n\n")

## 7. Tripulación (`Crew`) con proceso secuencial

In [9]:
via_fluida_crew = Crew(
    agents=[analista, optimizador, asesor_rutas, supervisor],
    tasks=[t_diagnostico, t_semaforos, t_rutas, t_reporte],
    process=Process.sequential,  # flujo lineal con 'context' explícito
    step_callback=registrar_paso,
    task_callback=registrar_tarea,
    verbose=True)

ENTRADAS = {
    "evento": "Salida de partido en el estadio + choque en el eje principal",
    "hora": "19:30 (sábado)", "clima": "lluvia moderada",
    "intersecciones": "INT-A, INT-B, INT-C, INT-D",
    "origen": "INT-A", "destino": "PERIF", "demanda_vph": 2400,
    "tramo_incidente": "INT-A|INT-B",
}

print("Crew creado:", len(via_fluida_crew.agents), "agentes,", len(via_fluida_crew.tasks), "tareas, proceso", via_fluida_crew.process)

Crew creado: 4 agentes, 4 tareas, proceso Process.sequential


## 8. Corrida de demostración
**Entradas:** sábado 19:30, lluvia moderada, salida de partido + choque en INT-A|INT-B (capacidad −50 %), 2400 veh/h de INT-A hacia PERIF.

In [10]:
import json
modo = os.getenv("VIAFLUIDA_LLM", "simulado")
print("Modelo de lenguaje:", modo)
print("Entradas:", json.dumps(ENTRADAS, ensure_ascii=False, indent=1))

_contador.update(herramientas=0, delegaciones=0)
with open(LOG_PASOS, "w", encoding="utf-8") as f:
    f.write(f"[Crew] Vía Fluida | process=sequential | agentes=4 | tareas=4 | llm={modo}\n")
    f.write(f"[Inputs] {json.dumps(ENTRADAS, ensure_ascii=False)}\n\n")

resultado = await via_fluida_crew.kickoff_async(inputs=ENTRADAS)

cierre = (f"[Crew] Ejecución finalizada | herramientas invocadas: "
          f"{_contador['herramientas']} | delegaciones: {_contador['delegaciones']}")
with open(LOG_PASOS, "a", encoding="utf-8") as f:
    f.write(cierre + "\n")
print("\n" + cierre)

Modelo de lenguaje: simulado
Entradas: {
 "evento": "Salida de partido en el estadio + choque en el eje principal",
 "hora": "19:30 (sábado)",
 "clima": "lluvia moderada",
 "intersecciones": "INT-A, INT-B, INT-C, INT-D",
 "origen": "INT-A",
 "destino": "PERIF",
 "demanda_vph": 2400,
 "tramo_incidente": "INT-A|INT-B"
}


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b6ae7425-f166-4ebd-8167-5a3f4ef72ea8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Evento: Salida de partido en el estadio + choque en el eje principal. Hora: 19:30 (sábado). Clima:       │
│  lluvia moderada. Con tus herramientas, consulta los sensores y calcula el TTI de TODAS las intersecciones      │
│  (INT-A, INT-B, INT-C, INT-D). Clasifica la severidad, identifica el acceso crítico de cada una y ordénalas de  │
│  la más a la menos crítica. No propongas acciones.                                                              │
│  ID: 295f2cce-060d-4981-9346-663c423c9586                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista de Percepción Vial                                                                             │
│                                                                                                                 │
│  Task: Evento: Salida de partido en el estadio + choque en el eje principal. Hora: 19:30 (sábado). Clima:       │
│  lluvia moderada. Con tus herramientas, consulta los sensores y calcula el TTI de TODAS las intersecciones      │
│  (INT-A, INT-B, INT-C, INT-D). Clasifica la severidad, identifica el acceso crítico de cada una y ordénalas de  │
│  la más a la menos crítica. No propongas acciones.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: leer_sensores_interseccion                                                                               │
│  Args: {"id_interseccion": "TODAS"}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {"INT-A": {"nombre": "Acceso norte zona estadio", "ciclo_s": 90, "verde_actual": {"NS": 41, "EO": 41},         │
│  "accesos": {"N": {"cola": 38, "vel": 6, "espera": 142, "flujo": 2600, "carriles": 2}, "S": {"cola": 12,        │
│  "vel": 18, "espera": 55, "flujo": 700, "carriles": 2}, "E": {"cola": 20, "vel": 11, "espera": 88, "flujo":     │
│  900, "carriles": 2}, "O": {"cola": 9, "vel": 22, "espera": 40, "flujo": 500, "carriles": 1}}}, "INT-B":        │
│  {"nombre": "Eje principal (aguas abajo del choque)", "ciclo_s": 90, "verde_actual": {"NS": 41, "EO": 41},      │
│  "accesos": {"N": {"cola": 6, "vel": 24, "espera": 35, "flujo": 900, "carriles": 2}, "S": {"cola": 8, "vel":    │
│  21, "espera": 40, "flujo": 1000, "carriles": 2}, "E": {"cola": 11, "vel": 15, "espera": 62, "flujo": 1500,     │
│  "carriles": 2}, "O": {"cola": 14, "vel": 13, "espera": 70, "flujo": 1700, "carriles": 2}}}, "INT-C":           │
│  {"nombre": "Alterna oriente", "ciclo_s": 90, "verde_actual": {"NS": 41, "EO": 41}, "accesos": {"N": {"cola":   │
│  3, "vel": 30, "espera": 22, "flujo": 600, "carriles": 1}, "S": {"cola": 4, "vel": 29, "espera": 24, "flujo":   │
│  650, "carriles": 1}, "E": {"cola": 2, "vel": 33, "espera": 18, "flujo": 500, "carriles": 1}, "O": {"cola": 3,  │
│  "vel": 31, "espera": 20, "flujo": 550, "carriles": 1}}}, "INT-D": {"nombre": "Alterna sur / conector",         │
│  "ciclo_s": 90, "verde_actual": {"NS": 41, "EO": 41}, "accesos": {"N": {"cola": 4, "vel": 28, "espera": 26,     │
│  "flujo": 700, "carriles": 1}, "S": {"cola": 5, "vel": 27, "espera": 28, "flujo": 720, "carriles": 1}, "E":     │
│  {"cola": 3, "vel": 31, "espera": 21, "flujo": 500, "carriles": 1}, "O": {"cola": 4, "vel": 29, "espera": 25,   │
│  "flujo": 560, "carriles": 1}}}}                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (47 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Supervisor y Sintetizador de Movilidad                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Reporte Vía Fluida - 19:30 (sábado), lluvia moderada                                                         │
│                                                                                                                 │
│  ## 1. Resumen ejecutivo                                                                                        │
│  Salida de partido en el estadio + choque en el eje principal (tramo INT-A|INT-B, capacidad -50 %). Con las     │
│  acciones propuestas, la demora estimada en las intersecciones intervenidas baja de 94.1 s a 34.7 s (-63.1 %)   │
│  y el tiempo medio INT-A -> PERIF de 11.6 a 6.1 min (-47.4 %). Meta PEAS (>= 15 %): se cumple según el modelo.  │
│                                                                                                                 │
│  ## 2. Diagnóstico                                                                                              │
│  | Intersección | TTI | Severidad | Acceso crítico (cola) |                                                     │
│  |---|---|---|---|                                                                                              │
│  | INT-A | 4.91 | Crítico | N (38 veh) |                                                                        │
│  | INT-B | 2.48 | Severo | O (14 veh) |                                                                         │
│  | INT-D | 1.4 | Moderado | S (5 veh) |                                                                         │
│  | INT-C | 1.31 | Moderado | S (4 veh) |                                                                        │
│                                                                                                                 │
│  ## 3. Acciones semafóricas                                                                                     │
│  - INT-A: ciclo 90 -> 120 s; verde NS 41 -> 81 s, EO 41 -> 31 s.                                                │
│  - INT-B: ciclo 90 -> 120 s; verde NS 41 -> 41 s, EO 41 -> 71 s.                                                │
│  - Ola verde INT-A -> INT-B: offset 15.0 s.                                                                     │
│                                                                                                                 │
│  ## 4. Rutas y mensaje a conductores                                                                            │
│  - 60 % por INT-A > INT-B > PERIF (6.2 min).                                                                    │
│  - 40 % por INT-A > INT-C > INT-D > PERIF (6.0 min).                                                            │
│  - Efecto manada: si el 100 % tomara la alterna, 16.0 min.                                                      │
│  - Panel: "CHOQUE EN EJE PRINCIPAL. ALTERNA ORIENTE-SUR HABILITADA."                                            │
│                                                                                                                 │
│  ## 5. Validación PEAS                                                                                          │
│  | Métrica | Antes | Después | Reducción | Meta |                                                               │
│  |---|---|---|---|---|                                                                                          │
│  | Demora media ponderada (s) | 94.1 | 34.7 | 63.1 % | 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Consolida los resultados de las tres tareas previas. Valida con tu herramienta la meta PEAS (>=15 % de   │
│  reducción de espera y de viaje). Verifica que ninguna ruta recomendada atraviese una intersección que el plan  │
│  semafórico deje sin mejora. Si la meta no se cumple o hay contradicciones, DELEGA la corrección al agente      │
│  responsable y reintegra.                                                                                       │
│  Agent: Supervisor y Sintetizador de Movilidad                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b6ae7425-f166-4ebd-8167-5a3f4ef72ea8                                                                       │
│  Final Output: # Reporte Vía Fluida - 19:30 (sábado), lluvia moderada                                           │
│                                                                                                                 │
│  ## 1. Resumen ejecutivo                                                                                        │
│  Salida de partido en el estadio + choque en el eje principal (tramo INT-A|INT-B, capacidad -50 %). Con las     │
│  acciones propuestas, la demora estimada en las intersecciones intervenidas baja de 94.1 s a 34.7 s (-63.1 %)   │
│  y el tiempo medio INT-A -> PERIF de 11.6 a 6.1 min (-47.4 %). Meta PEAS (>= 15 %): se cumple según el modelo.  │
│                                                                                                                 │
│  ## 2. Diagnóstico                                                                                              │
│  | Intersección | TTI | Severidad | Acceso crítico (cola) |                                                     │
│  |---|---|---|---|                                                                                              │
│  | INT-A | 4.91 | Crítico | N (38 veh) |                                                                        │
│  | INT-B | 2.48 | Severo | O (14 veh) |                                                                         │
│  | INT-D | 1.4 | Moderado | S (5 veh) |                                                                         │
│  | INT-C | 1.31 | Moderado | S (4 veh) |                                                                        │
│                                                                                                                 │
│  ## 3. Acciones semafóricas                                                                                     │
│  - INT-A: ciclo 90 -> 120 s; verde NS 41 -> 81 s, EO 41 -> 31 s.                                                │
│  - INT-B: ciclo 90 -> 120 s; verde NS 41 -> 41 s, EO 41 -> 71 s.                                                │
│  - Ola verde INT-A -> INT-B: offset 15.0 s.                                                                     │
│                                                                                                                 │
│  ## 4. Rutas y mensaje a conductores                                                                            │
│  - 60 % por INT-A > INT-B > PERIF (6.2 min).                                                                    │
│  - 40 % por INT-A > INT-C > INT-D > PERIF (6.0 min).                                                            │
│  - Efecto manada: si el 100 % tomara la alterna, 16.0 min.                                                      │
│  - Panel: "CHOQUE EN EJE PRINCIPAL. ALTERNA ORIENTE-SUR HABILITADA."                                            │
│                                                                                                                 │
│  ## 5. Validación PEAS                                                                                          │
│  | Métrica | Antes | Después | Reducción | Meta |                                                               │
│  |---|---|---|---|---|                                                                                          │
│  | Demora media ponderada (s) | 94.1 | 34.7 | 63.1 % |


[Crew] Ejecución finalizada | herramientas invocadas: 11 | delegaciones: 0


### 8.1 Log de razonamiento e interacción entre agentes

In [11]:
print(open(LOG_PASOS, encoding="utf-8").read())

[Crew] Vía Fluida | process=sequential | agentes=4 | tareas=4 | llm=simulado
[Inputs] {"evento": "Salida de partido en el estadio + choque en el eje principal", "hora": "19:30 (sábado)", "clima": "lluvia moderada", "intersecciones": "INT-A, INT-B, INT-C, INT-D", "origen": "INT-A", "destino": "PERIF", "demanda_vph": 2400, "tramo_incidente": "INT-A|INT-B"}

Thought: Primero leo los sensores de toda la red y después calculo el TTI de cada intersección.
## Using tool: leer_sensores_interseccion
## Tool Input: {"id_interseccion": "TODAS"}
## Tool Output: {"INT-A": {"nombre": "Acceso norte zona estadio", "ciclo_s": 90, "verde_actual": {"NS": 41, "EO": 41}, "accesos": {"N": {"cola": 38, "vel": 6, "espera": 142, "flujo": 2600, "carriles": 2}, "S": {"cola": 12, "vel": 18, "espera": 55, "flujo": 700, "carriles": 2}, "E": {"cola": 20, "vel": 11, "espera": 88, "flujo": 900, "carriles": 2}, "O": {"cola": 9, "vel": 22, "espera": 40, "flujo": 500, "carriles": 1}}}, "INT-B": {"nombre": "Eje principal 

### 8.2 Respuesta final consolidada del Crew

In [12]:
from IPython.display import Markdown, display
display(Markdown(resultado.raw))

# Reporte Vía Fluida - 19:30 (sábado), lluvia moderada

## 1. Resumen ejecutivo
Salida de partido en el estadio + choque en el eje principal (tramo INT-A|INT-B, capacidad -50 %). Con las acciones propuestas, la demora estimada en las intersecciones intervenidas baja de 94.1 s a 34.7 s (-63.1 %) y el tiempo medio INT-A -> PERIF de 11.6 a 6.1 min (-47.4 %). Meta PEAS (>= 15 %): se cumple según el modelo.

## 2. Diagnóstico
| Intersección | TTI | Severidad | Acceso crítico (cola) |
|---|---|---|---|
| INT-A | 4.91 | Crítico | N (38 veh) |
| INT-B | 2.48 | Severo | O (14 veh) |
| INT-D | 1.4 | Moderado | S (5 veh) |
| INT-C | 1.31 | Moderado | S (4 veh) |

## 3. Acciones semafóricas
- INT-A: ciclo 90 -> 120 s; verde NS 41 -> 81 s, EO 41 -> 31 s.
- INT-B: ciclo 90 -> 120 s; verde NS 41 -> 41 s, EO 41 -> 71 s.
- Ola verde INT-A -> INT-B: offset 15.0 s.

## 4. Rutas y mensaje a conductores
- 60 % por INT-A > INT-B > PERIF (6.2 min).
- 40 % por INT-A > INT-C > INT-D > PERIF (6.0 min).
- Efecto manada: si el 100 % tomara la alterna, 16.0 min.
- Panel: "CHOQUE EN EJE PRINCIPAL. ALTERNA ORIENTE-SUR HABILITADA."

## 5. Validación PEAS
| Métrica | Antes | Después | Reducción | Meta |
|---|---|---|---|---|
| Demora media ponderada (s) | 94.1 | 34.7 | 63.1 % | >= 15 % |
| Tiempo medio de viaje (min) | 11.6 | 6.1 | 47.4 % | >= 15 % |
- Consistencia semáforos-rutas: las rutas recomendadas solo cruzan intersecciones que el plan mejora (INT-A, INT-B). No fue necesario delegar.

## 6. Riesgos y siguientes pasos
- Cifras de un modelo simplificado con datos sintéticos: validar en simulación (SUMO) y piloto.
- Vigilar INT-D (TTI 1.4), INT-C (TTI 1.31) por posible saturación de la alterna.
- Reevaluar cada 5 min; restaurar planes al despejar el choque.

**Lectura del resultado:** con datos sintéticos y un modelo simplificado, la demora en las intersecciones intervenidas baja 63.1 % y el tiempo de viaje 47.4 %, por encima de la meta PEAS de 15 %. Mandar a todos por la alterna (efecto manada) sería peor que no intervenir (16.0 vs 11.6 min); repartir 60/40 lo baja a 6.1 min.